<a href="https://colab.research.google.com/github/arnabd64/Google-Colab-Notebooks/blob/main/Amazon_Reviews_Multilinugual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon Reviews Classifier using BERT

## Install dependencies

In [ ]:
! pip install --progress-bar off \
    transformers[torch] \
    evaluate \
    datasets \
> install.log

## Login to Huggingface

In [ ]:
from google.colab import drive, userdata
import huggingface_hub

huggingface_hub.login(userdata.get("HF"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


## Import Dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATASET = "mteb/amazon_reviews_multi"

dataset = concatenate_datasets([
    load_dataset(DATASET, "all_languages", split="train"),
    load_dataset(DATASET, "all_languages", split="test"),
    load_dataset(DATASET, "all_languages", split="validation")
])

dataset = dataset.train_test_split(test_size=0.01, shuffle=True)

display(dataset)

Extracting data files:   0%|          | 0/6 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/6 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/6 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 1247400
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 12600
    })
})

## Define Labels for Classification

In [ ]:
LABELS = ["1-STAR", "2-STARS", "3-STARS", "4-STARS", "5-STARS"]
ID2LABEL = {idx:value for idx, value in enumerate(LABELS)}
LABEL2ID = {value:idx for idx, value in enumerate(LABELS)}

## Initiate base model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
get_model = lambda model_name: AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

## Preprocessing

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'],
                     truncation = True,
                     padding = "max_length",
                     max_length = 512,
                     return_tensors = "pt")


dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(['text'])

Map:   0%|          | 0/1247400 [00:00<?, ? examples/s]

Map:   0%|          | 0/12600 [00:00<?, ? examples/s]

## Evaluation Function

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
cola = evaluate.load("glue", "cola")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    accuracy_metric = accuracy.compute(predictions=predictions, references=labels)
    cola_metric = cola.compute(predictions=predictions, references=labels)

    return {"accuracy": accuracy_metric["accuracy"], "matthews_correlation": cola_metric["matthews_correlation"]}

## Setup Trainer module & start Training

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir = "distilbert-base-amazon-multi",
    save_safetensors = True,
    save_strategy = "steps",
    push_to_hub = True,
    save_steps = 30_000,
    evaluation_strategy = "steps",
    eval_steps = 10_000,
    max_steps = 100_000,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 320,
    learning_rate = 5e-5,
    weight_decay = 1e-9,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    report_to = ['tensorboard'],
    load_best_model_at_end = True,
    metric_for_best_model = "matthews_correlation",
    greater_is_better = True
)

trainer = Trainer(
    model = get_model(MODEL),
    args = args,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    compute_metrics = compute_metrics
)

output = trainer.train()

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Accuracy,Matthews Correlation
10000,1.000800,1.002674,0.561587,0.452019
20000,0.954500,0.970487,0.580952,0.478759
30000,0.921600,0.941458,0.588254,0.486816
40000,0.876500,0.949482,0.589127,0.487143
50000,0.883700,0.925361,0.599206,0.499654
60000,0.875300,0.919881,0.601429,0.502854
70000,0.857200,0.910799,0.608968,0.511722
80000,0.785100,0.927586,0.605238,0.506578
90000,0.791800,0.929231,0.605476,0.507203
100000,0.793000,0.928799,0.606429,0.508379


## Save Metrics and Push to Hub

In [ ]:
trainer.save_metrics(metrics=trainer.evaluate(dataset['test']), split="evaluate", combined=False)

In [ ]:
trainer.push_to_hub(dataset=DATASET, tags=["pytorch","amazon-rating","DistilBERTForSequenceClassification"])

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

events.out.tfevents.1701484697.4888c9987e8e.398.0:   0%|          | 0.00/40.6k [00:00<?, ?B/s]

events.out.tfevents.1701531798.4888c9987e8e.398.1:   0%|          | 0.00/482 [00:00<?, ?B/s]

'https://huggingface.co/arnabdhar/distilbert-base-amazon-multi/tree/main/'

In [ ]:
# ! rm -rfv distilbert-base-amazon-multi